In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score, precision_score, recall_score

from src.preprocess import (
    select_core_features,
    clean_features,
    build_preprocessor
)

df = pd.read_csv("../data/home-credit-default-risk/application_train.csv")

df = select_core_features(df)
df = clean_features(df)

X = df.drop(columns=["TARGET"])
y = df["TARGET"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

preprocessor = build_preprocessor()

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        solver="lbfgs"
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        random_state=42,
        class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )
}

results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_val)
    y_prob = pipeline.predict_proba(X_val)[:, 1]
    
    auc = roc_auc_score(y_val, y_prob)
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    cm = confusion_matrix(y_val, y_pred)
    
    results.append({
        "Model": name,
        "Train Accuracy": pipeline.score(X_train, y_train),
        "Validation Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "ROC-AUC": auc
    })
    
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print("Train Accuracy:", pipeline.score(X_train, y_train))
    print("Validation Accuracy:", accuracy)
    print("ROC-AUC Score:", auc)
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred, zero_division=0))
    print("Confusion Matrix:")
    print(cm)

results_df = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False)

print("\nFinal Model Comparison Table:")
print(results_df)

C:\Users\Student\miniconda3\envs\credit-risk-env\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Logistic Regression
Train Accuracy: 0.5934888296315567
Validation Accuracy: 0.5944750662569306
ROC-AUC Score: 0.6459785016748711

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.59      0.73     56538
           1       0.12      0.63      0.20      4965

    accuracy                           0.59     61503
   macro avg       0.53      0.61      0.46     61503
weighted avg       0.88      0.59      0.69     61503

Confusion Matrix:
[[33429 23109]
 [ 1832  3133]]

Decision Tree
Train Accuracy: 0.6708928164937725
Validation Accuracy: 0.6721623335447051
ROC-AUC Score: 0.716615810122554

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.67      0.79     56538
           1       0.15      0.65      0.24      4965

    accuracy                           0.67     61503
   macro avg       0.55      0.66      0.52     61503
weighted avg       0.89      0.67      0.75     615

In Phase 4, three models were compared: Logistic Regression, Decision Tree, and Random Forest. Random Forest achieved the best overall performance, with a ROC-AUC of 0.7391 and validation accuracy of 0.7192. Decision Tree also outperformed Logistic Regression, achieving a ROC-AUC of 0.7166. Logistic Regression showed the weakest performance and also produced a convergence warning, indicating that it may require additional tuning or scaling. Based on these results, Random Forest was selected as the final model for the next phase. However, Logistic Regression remains valuable as a baseline due to its interpretability.